# Data Processing

Included currently:

-   AGFD
-   UNSW
-   MCC5


In [2]:
from pathlib import Path

import numpy as np
import polars as pl
from scipy.io import loadmat

# NOTE: Optional stuff
# import pandas as pd
# import json

## Helpers


In [3]:
def calc_speed(tach, sampling_freq, key_pulse_ref_height=1):
    """
    Calculate speed in RPM from tachometer signal.
    Args:
        tach (np.ndarray): Tachometer signal array.
        sampling_freq (int, optional): Sampling frequency in Hz. Defaults to 200 kHz.
        key_pulse_ref_height (float, optional): Reference height to detect key pulses. Defaults to 1.
    Returns:
        float: Calculated speed in RPM.
    """

    key_pulses = np.where(
        np.logical_and(
            tach[:-1] <= key_pulse_ref_height, tach[1:] > key_pulse_ref_height
        )
    )[
        0
    ]  # Locations of key pulses

    # Only count the time between the first and last detected key pulse to get full rotations
    full_rotations_seconds = (
        len(tach[key_pulses[0] : key_pulses[-1]]) / sampling_freq
    )  # seconds
    # `-1` because n key pulses correspond to n-1 full rotations
    rot_speed = (len(key_pulses) - 1) / full_rotations_seconds  # Hz
    rot_speed_rpm = rot_speed * 60  # RPM
    return rot_speed_rpm


def AGFD_fix_time(X):
    X = X.to_numpy()
    increment = 1 / 3012  # For 3012 Hz sampling rate
    loc = np.where(X[1:] < X[:-1])[0] + 1  # + 1 needed to get the right index

    if len(loc) == 0:
        return X

    for l in loc:
        wrong = X[l - 1] - X[l]
        print(f"Wrong time at {l}")
        diff = np.concatenate(
            [
                np.zeros(l),
                np.ones(len(X) - l) * (increment + wrong),
            ]
        )
        X = X + diff

    return X

## UNSW

-   Mendeley Data: https://data.mendeley.com/datasets/p72x3m92cv/1

**Notes**

-   Sampling frequency: 200 kHz
-   "S" size crack doesn't really show any fault signs


In [ ]:
UNSW_raw_data_dir = Path.cwd().parent.parent.parent / "data" / "UNSW_gear_crack"
UNSW_processed_data_dir = Path.cwd().parent.parent.parent / "data" / "ROTOMATE" / "UNSW"
UNSW_processed_data_dir.mkdir(parents=True, exist_ok=True)  # Ensure directory exists

### Processing


In [ ]:
UNSW_sampling_rate = 200000  # Hz
include_Yaxins = False  # Data not in published dataset, which I put in the same folder as the public data. Can be removed by others.

files = UNSW_raw_data_dir.glob("*.mat")

UNSW_measurements = []
for f in files:
    print(f)

    # Get measurement specs
    fname_parts = f.stem.split("_")

    speed_nominal = int(fname_parts[1][:2])
    load_nominal = int(fname_parts[2][:2])
    severity = fname_parts[3]

    # Load data
    non_OT_mat = loadmat(f)

    try:
        torqueAvg = non_OT_mat["torqueAvg"][
            0, 0
        ]  # From actual measurements, differs slightly from nominal load
        tacho_in = non_OT_mat["signals"][:, 0]
        speed = calc_speed(tacho_in, UNSW_sampling_rate, key_pulse_ref_height=1)
        signal = non_OT_mat["signals"][:, 4]  # Accelerometer signal

        df = pl.DataFrame(
            {
                "severity": severity,
                "speed_nominal": speed_nominal,
                "load_nominal": load_nominal,
                "speed": calc_speed(tacho_in),
                "load": torqueAvg,
                "tacho_in": tacho_in,
                "signal": signal,
                # NOTE: Add these if needed
                # "encoder_in": non_OT_mat["signals"][:, 1],
                # "tacho_out": non_OT_mat["signals"][:, 2],
                # "encoder_out": non_OT_mat["signals"][:, 3],
            }
        )
        df = df.with_columns(pl.col("speed").cast(pl.Float32))
    # NOTE: For data not in the published dataset, can be removed by others
    except KeyError:
        if not include_Yaxins:
            print(f"{f} is Yaxin's data, skipping...")
            continue
        else:
            raise Exception("Yaxin's data inclusion not implemented yet.")

    UNSW_measurements.append(df)

UNSW_measurements = pl.concat(UNSW_measurements)
UNSW_measurements.head(2)

### Write & Read

#### Parquet

Simple and works when you want to read full data.


In [ ]:
# Write

# Compressed by default
UNSW_measurements.write_parquet(
    UNSW_processed_data_dir / "UNSW_gear_crack.parquet",
)

In [ ]:
# Read

UNSW_df_parquet = pl.read_parquet(
    UNSW_processed_data_dir / "UNSW_gear_crack.parquet",
)

# NOTE: With Pandas
# UNSW_df_parquet = pd.read_parquet(
#     UNSW_processed_data_dir / "UNSW_gear_crack.parquet",
# )

#### Arrow

Same as Parquet but in different flavour. Slightly slower write and read.


In [ ]:
# Write

UNSW_measurements.write_ipc(
    UNSW_processed_data_dir / "UNSW_gear_crack.arrow",
    compression="zstd",  # Drops file size significantly (~25 Gb to ~3 Gb)
)

In [ ]:
# Read

UNSW_df_arrow = pl.read_ipc(
    UNSW_processed_data_dir / "UNSW_gear_crack.arrow",
    memory_map=False,  # Doesn't work with compressed data
)

# NOTE: With Pandas
# UNSW_df_arrow = pd.read_feather(
#     UNSW_processed_data_dir / "UNSW_gear_crack.arrow",
# )

#### Parquet Hive

Fast to read when you only want part of the data.


In [ ]:
# Write

UNSW_measurements.write_parquet(
    UNSW_processed_data_dir / "UNSW_gear_crack_hive",
    partition_by=["severity", "speed_nominal", "load_nominal"],
)

In [ ]:
# Read


def read_UNSW_hive(
    data_location,
    severity=None,
    speed_nominal=None,
    load_nominal=None,
    drop_nominal=True,
):
    """
    data_location: Path object pointing to the hive data directory
    severity options: "H", "S", "M", "L"
    speed_nominal_options: 2, 5, 10, 15, 20
    load_nominal_options: 0, 5, 10, 15, 20
    """

    if isinstance(data_location, pl.DataFrame):
        df_hive = data_location
        collect = False
    elif isinstance(data_location, Path):
        hive_files = list(data_location.glob("**/*.parquet"))
        df_hive = pl.scan_parquet(hive_files, hive_partitioning=True)
    else:
        raise ValueError("data_location must be a polars DataFrame or a Path object")

    df_hive = pl.scan_parquet(
        hive_files,
        hive_partitioning=True,
    )

    # Get only required partitions
    if severity is not None:
        df_hive = df_hive.filter(
            pl.col("severity") == severity,
        )

    if speed_nominal is not None:
        df_hive = df_hive.filter(
            pl.col("speed_nominal") == speed_nominal,
        )

    if load_nominal is not None:
        df_hive = df_hive.filter(
            pl.col("load_nominal") == load_nominal,
        )

    # Drop nominal values
    if drop_nominal:
        df_hive = df_hive.select(pl.exclude(["load_nominal", "speed_nominal"]))

    if collect:
        df_hive = df_hive.collect()

    return df_hive

In [ ]:
# Read everything

read_UNSW_hive(UNSW_processed_data_dir / "UNSW_gear_crack_hive")

# NOTE: Read from single file
# read_UNSW_hive(UNSW_df_parquet)

In [ ]:
# Get one thing

read_UNSW_hive(
    UNSW_processed_data_dir / "UNSW_gear_crack_hive",
    severity="H",
    speed_nominal=10,
    load_nominal=10,
).head(3)

In [ ]:
# Loop through all

for severity in ["H", "S", "M", "L"]:
    for speed_nominal in [2, 5, 10, 15, 20]:
        for load_nominal in [0, 5, 10, 15, 20]:
            measurement = read_UNSW_hive(
                UNSW_processed_data_dir / "UNSW_gear_crack_hive",
                severity=severity,
                speed_nominal=speed_nominal,
                load_nominal=load_nominal,
            )

            # NOTE: Do something with `measurement` here

### JSON version

Very slow and no option to compress data.


In [ ]:
include_Yaxins = False  # Experiments by Yaxin (wear / no lubrication)

output_data_folder = UNSW_processed_data_dir / "UNSW_json"
Path(output_data_folder).mkdir(parents=True, exist_ok=True)

files = UNSW_raw_data_dir.glob("*.mat")

measurements = []
for f in files:
    print(f)

    # Get measurement specs
    fname_parts = f.stem.split("_")

    speed_nominal = int(fname_parts[1][:2])
    load_nominal = int(fname_parts[2][:2])
    severity = fname_parts[3]

    non_OT_mat = loadmat(f)

    try:
        torqueAvg = non_OT_mat["torqueAvg"][0, 0]
        speed = calc_speed(tacho_in, UNSW_sampling_rate, key_pulse_ref_height=1)
        tacho_in = non_OT_mat["signals"][:, 0]
        signal = non_OT_mat["signals"][:, 4]

        measurement = {
            "severity": severity,
            "speed": speed,
            "load": torqueAvg,
            "tacho_in": tacho_in,
            "signal": signal,
        }

        with open(
            output_data_folder / f"{speed_nominal}Hz_{load_nominal}Nm_{severity}.json",
            "w",
        ) as fp:
            json.dump(
                {
                    k: (v.tolist() if hasattr(v, "tolist") else v)
                    for k, v in measurement.items()
                },
                fp,
            )

    except KeyError:
        if not include_Yaxins:
            print(f"{f} is Yaxin's data, skipping...")
            continue
        else:
            raise Exception("Yaxin's data inclusion not implemented yet.")

## MCC5

-   Journal paper: https://www.sciencedirect.com/science/article/pii/S2352340924004220
-   Data in Brief: https://arxiv.org/pdf/2403.12521
-   Mendeley Data: https://data.mendeley.com/datasets/p92gj2732w/2
-   GitHub\_ https://github.com/liuzy0708/MCC5-THU-Gearbox-Benchmark-Datasets

**Known problems**

_Suffix `ERROR_` added to these files to make skipping easier\_

-   gear_pitting_M_speed_circulation_20Nm-1000rpm.csv - Distortion, gain error, or something
-   gear_pitting_M_torque_circulation_2000rpm_10Nm.csv - Distortion, gain error, or something

**Notes**

-   Sampling frequency: 12.8 kHz
-   `gearbox_vibration_x` seems to work the best
-   Speeds seem to be +/- 50 RPM from the reported ones, but highest discrepancy is almost 150 RPM (at motor).


In [4]:
MCC5_raw_data_dir = Path.cwd().parent.parent.parent / "data" / "MCC5-THU"
MCC5_processed_data_dir = Path.cwd().parent.parent.parent / "data" / "ROTOMATE" / "MCC5"
MCC5_processed_data_dir.mkdir(parents=True, exist_ok=True)  # Ensure directory exists

### Processing


In [5]:
MCC5_sampling_rate = 12800  # Hz
MCC5_first_gear_ratio = 29 / 95  # Ratio of first gear in the drivetrain

MCC5_speed_cycle_map = {
    1000: 500,
    2000: 1500,
    3000: 2500,
}

MCC5_load_cycle_map = {
    10: 5,
    20: 15,
}

MCC5_fault_map = {
    "teeth_crack": "crack",
    "gear_wear": "wear",
    "gear_pitting": "pitting",
    "miss_teeth": "missing_tooth",
    "teeth_break": "broken_tooth",
}

MCC5_severity_map = {
    "L": "S",
    "M": "M",
    "H": "L",
    "XL": "XL",
}

# Seconds to cut out for accelerations and deaccelerations
MCC5_time_margin = 0.5

# Raw data
files = MCC5_raw_data_dir.glob("*.csv")

MCC5_measurements = []
for f in files:
    print(f)

    # Get measurement specs
    p = f.stem.split("_")

    # Skip files with something wrong in them
    if p[0] == "ERROR":
        print(f"Skipping {f} due to ERROR in filename")
        continue

    # Parse measurement specs
    ##

    # Healthy
    if p[0] == "health":
        fault = "healthy"
        severity = "-"

        # The rest depend on which condition is being cycled
        if f"{p[1]}_{p[2]}" == "torque_circulation":
            circulation = "torque"
            speed_nominal = int(p[3].replace("rpm", ""))
            load_nominal = int(p[4].replace("Nm", ""))
        elif f"{p[1]}_{p[2]}" == "speed_circulation":
            circulation = "speed"
            load_nominal, speed_nominal = p[3].split("-")
            load_nominal = int(load_nominal.replace("Nm", ""))
            speed_nominal = int(speed_nominal.replace("rpm", ""))
    # Faulty
    else:
        fault = f"{p[0]}_{p[1]}"

        # NOTE: Multi-fault measurements are skipped for now
        if fault not in MCC5_fault_map.keys() or p[2] == "and":
            continue

        # Rename some faults for consistency
        fault = MCC5_fault_map[fault]
        # Missing tooth doesn't have severity. Use "H" (maps to "L" [i.e. large])
        if fault == "missing_tooth":
            p.insert(2, "H")
        # Rename severities for consistency
        severity = MCC5_severity_map[p[2]]  # L/M/H -> S/M/L

        # The rest depend on which condition is cycled
        if f"{p[3]}_{p[4]}" == "torque_circulation":
            circulation = "load"
            speed_nominal = int(p[5].replace("rpm", ""))
            load_nominal = int(p[6].replace("Nm", ""))
        elif f"{p[3]}_{p[4]}" == "speed_circulation":
            circulation = "speed"
            load_nominal, speed_nominal = p[5].split("-")
            load_nominal = int(load_nominal.replace("Nm", ""))
            speed_nominal = int(speed_nominal.replace("rpm", ""))
        else:
            raise Exception(f"Cannot parse circulation from filename: {f}")

    print(fault, severity, circulation, speed_nominal, load_nominal)

    # Signal
    ##

    df = pl.read_csv(f)

    tacho = df.select("speed").to_numpy().flatten()

    tach_part_1 = tacho[
        int((10 + MCC5_time_margin) * MCC5_sampling_rate) : int(
            (20 - MCC5_time_margin) * MCC5_sampling_rate
        )
    ]
    tach_part_2 = tacho[
        int((25 + MCC5_time_margin) * MCC5_sampling_rate) : int(
            (35 - MCC5_time_margin) * MCC5_sampling_rate
        )
    ]
    tach_part_3 = tacho[
        int((40 + MCC5_time_margin) * MCC5_sampling_rate) : int(
            (50 - MCC5_time_margin) * MCC5_sampling_rate
        )
    ]
    tachs_in = [tach_part_1, tach_part_2, tach_part_3]

    # Calculate real speeds
    speed_1 = calc_speed(
        tach_part_1, sampling_freq=MCC5_sampling_rate, key_pulse_ref_height=2
    )
    speed_2 = calc_speed(
        tach_part_2, sampling_freq=MCC5_sampling_rate, key_pulse_ref_height=2
    )
    speed_3 = calc_speed(
        tach_part_3, sampling_freq=MCC5_sampling_rate, key_pulse_ref_height=2
    )
    speeds = [speed_1, speed_2, speed_3]

    # There is a gear ratio between motor and gearbox speeds
    for i in range(3):
        speeds[i] = speeds[i] * MCC5_first_gear_ratio  # Gear ratio

    sensor_signals = [{}, {}, {}]
    for sensor in [
        "motor_vibration_x",
        "motor_vibration_y",
        "motor_vibration_z",
        "gearbox_vibration_x",
        "gearbox_vibration_y",
        "gearbox_vibration_z",
    ]:
        signal = df.select(sensor).to_numpy().flatten()
        signal_part_1 = signal[
            int((10 + MCC5_time_margin) * MCC5_sampling_rate) : int(
                (20 - MCC5_time_margin) * MCC5_sampling_rate
            )
        ]
        signal_part_2 = signal[
            int((25 + MCC5_time_margin) * MCC5_sampling_rate) : int(
                (35 - MCC5_time_margin) * MCC5_sampling_rate
            )
        ]
        signal_part_3 = signal[
            int((40 + MCC5_time_margin) * MCC5_sampling_rate) : int(
                (50 - MCC5_time_margin) * MCC5_sampling_rate
            )
        ]

        sensor_signals[0][sensor] = signal_part_1
        sensor_signals[1][sensor] = signal_part_2
        sensor_signals[2][sensor] = signal_part_3

    df = []
    for i in range(3):
        speed_nominal_corrected = speed_nominal
        load_nominal_corrected = load_nominal
        # Middle section has a different speed/load from the nominal value due to cycling
        if i == 1:
            if circulation == "speed":
                speed_nominal_corrected = MCC5_speed_cycle_map[speed_nominal]
            elif circulation == "load":
                load_nominal_corrected = MCC5_load_cycle_map[load_nominal]

        df.append(
            pl.DataFrame(
                {
                    "fault": fault,
                    "severity": severity,
                    "circulation": circulation,
                    "section": i,
                    "speed_nominal": speed_nominal_corrected,
                    "load_nominal": load_nominal_corrected,
                    "speed": speeds[i],
                    "tacho_in": tachs_in[i],
                    **sensor_signals[i],
                }
            )
        )

    # Make DF of one file
    df = pl.concat(df)
    MCC5_measurements.append(df)

# Combine files
MCC5_measurements = pl.concat(MCC5_measurements)
MCC5_measurements.head(3)

/Users/hamalaa14/git/data/MCC5-THU/teeth_break_M_speed_circulation_10Nm-1000rpm.csv
broken_tooth M speed 1000 10
/Users/hamalaa14/git/data/MCC5-THU/teeth_break_M_torque_circulation_1000rpm_20Nm.csv
broken_tooth M load 1000 20
/Users/hamalaa14/git/data/MCC5-THU/teeth_crack_L_torque_circulation_1000rpm_20Nm.csv
crack S load 1000 20
/Users/hamalaa14/git/data/MCC5-THU/teeth_break_and_bearing_outer_M_torque_circulation_1000rpm_20Nm.csv
/Users/hamalaa14/git/data/MCC5-THU/gear_pitting_H_torque_circulation_2000rpm_10Nm.csv
pitting L load 2000 10
/Users/hamalaa14/git/data/MCC5-THU/teeth_break_and_bearing_inner_H_speed_circulation_20Nm-1000rpm.csv
/Users/hamalaa14/git/data/MCC5-THU/teeth_break_L_speed_circulation_20Nm-1000rpm.csv
broken_tooth S speed 1000 20
/Users/hamalaa14/git/data/MCC5-THU/gear_wear_H_speed_circulation_10Nm-1000rpm.csv
wear L speed 1000 10
/Users/hamalaa14/git/data/MCC5-THU/teeth_crack_H_torque_circulation_2000rpm_10Nm.csv
crack L load 2000 10
/Users/hamalaa14/git/data/MCC5-T

fault,severity,circulation,section,speed_nominal,load_nominal,speed,tacho_in,motor_vibration_x,motor_vibration_y,motor_vibration_z,gearbox_vibration_x,gearbox_vibration_y,gearbox_vibration_z
str,str,str,i32,i32,i32,f64,f64,f64,f64,f64,f64,f64,f64
"""broken_tooth""","""M""","""speed""",0,1000,10,299.258139,0.002265,-0.45985,-0.278449,0.330734,-0.895619,-0.507688,-0.301957
"""broken_tooth""","""M""","""speed""",0,1000,10,299.258139,0.002726,-0.524867,-0.310183,0.2964497,-0.522768,-0.563371,-0.216067
"""broken_tooth""","""M""","""speed""",0,1000,10,299.258139,0.007272,-0.318456,-0.162256,0.0850439,-0.173044,-0.260949,0.0344515


### Write & Read


#### Parquet

Simple and works when you want to read full data.

Arrow / Feather and Pandas equivalents commented out.


In [5]:
MCC5_measurements.head(2)

fault,severity,circulation,section,speed_nominal,load_nominal,speed,tach_in,motor_vibration_x,motor_vibration_y,motor_vibration_z,gearbox_vibration_x,gearbox_vibration_y,gearbox_vibration_z
str,str,str,i32,i32,i32,f64,f64,f64,f64,f64,f64,f64,f64
"""broken_tooth""","""M""","""speed""",0,1000,10,299.258139,0.002265,-0.45985,-0.278449,0.330734,-0.895619,-0.507688,-0.301957
"""broken_tooth""","""M""","""speed""",0,1000,10,299.258139,0.002726,-0.524867,-0.310183,0.2964497,-0.522768,-0.563371,-0.216067


In [6]:
# Write

# Compressed by default
MCC5_measurements.write_parquet(
    MCC5_processed_data_dir / "MCC5.parquet",
)

# NOTE: To Arrow / Feather
# MCC5_measurements.write_ipc(
#     MCC5_processed_data_dir / "MCC5.arrow",
#     compression="zstd",  # Drops file size significantly (~25 Gb to ~3 Gb)
# )

In [ ]:
# Read

MCC5_df_parquet = pl.read_parquet(
    MCC5_processed_data_dir / "MCC5.parquet",
)

# NOTE: With Pandas
# MCC5_df_parquet = pd.read_parquet(
#     MCC5_processed_data_dir / "MCC5.parquet",
# )

# NOTE: From Arrow / Feather
# MCC5_df_arrow = pl.read_ipc(
#     MCC5_processed_data_dir / "MCC5.arrow",
#     memory_map=False,  # Doesn't work with compressed data
# )


# NOTE: From Arrow / Feather with Pandas
# MCC5_df_arrow = pd.read_feather(
#     MCC5_processed_data_dir / "MCC5_gear_crack.arrow",
# )

#### Parquet Hive

Fast to read when you only want part of the data.


In [7]:
# Write

MCC5_measurements.write_parquet(
    MCC5_processed_data_dir / "MCC5_hive",
    partition_by=["speed_nominal"],
)

In [ ]:
# Read


def read_MCC5_hive(
    data_location,
    fault=None,
    severity=None,
    circulation=None,
    section=None,
    speed_nominal=None,
    load_nominal=None,
    drop_nominal=True,
):
    """
    data_location: Path object pointing to the hive data directory
    severity options: "H", "S", "M", "L"
    speed_nominal_options: 2, 5, 10, 15, 20
    load_nominal_options: 0, 5, 10, 15, 20
    """

    collect = True
    if isinstance(data_location, pl.DataFrame):
        df_hive = data_location
        collect = False
    elif isinstance(data_location, Path):
        hive_files = list(data_location.glob("**/*.parquet"))
        df_hive = pl.scan_parquet(hive_files, hive_partitioning=True)
    else:
        raise ValueError("data_location must be a polars DataFrame or a Path object")

    # Get only required partitions
    if fault is not None:
        df_hive = df_hive.filter(
            pl.col("fault") == fault,
        )

    if severity is not None:
        df_hive = df_hive.filter(
            pl.col("severity") == severity,
        )

    if circulation is not None:
        df_hive = df_hive.filter(
            pl.col("circulation") == circulation,
        )

    if section is not None:
        df_hive = df_hive.filter(
            pl.col("section") == section,
        )

    if speed_nominal is not None:
        df_hive = df_hive.filter(
            pl.col("speed_nominal") == speed_nominal,
        )

    if load_nominal is not None:
        df_hive = df_hive.filter(
            pl.col("load_nominal") == load_nominal,
        )

    # Drop nominal values
    if drop_nominal:
        df_hive = df_hive.select(pl.exclude(["load_nominal", "speed_nominal"]))

    if collect:
        df_hive = df_hive.collect()

    return df_hive

In [ ]:
# Read everything

read_MCC5_hive(MCC5_processed_data_dir / "MCC5_hive").head(2)

In [ ]:
# Get one thing

read_MCC5_hive(
    MCC5_processed_data_dir / "MCC5_hive",
    # MCC5_df_parquet,
    fault="crack",
    severity="L",
    circulation="speed",
    speed_nominal=1000,
    load_nominal=10,
).head(2)

In [ ]:
# Loop through all

for fault in ["crack", "wear", "pitting", "missing_tooth", "broken_tooth"]:
    for severity in ["-", "S", "M", "L"]:
        for circulation in ["speed", "load"]:
            for speed_nominal in [500, 1000, 1500, 2000, 2500, 3000]:
                for load_nominal in [5, 10, 15, 20]:
                    measurement = read_MCC5_hive(
                        MCC5_processed_data_dir / "MCC5_hive",
                        severity=severity,
                        speed_nominal=speed_nominal,
                        load_nominal=load_nominal,
                    )
                    # NOTE: Do something with `measurement` here

## AGFD

-   Journal: https://www.sciencedirect.com/science/article/pii/S2352340924011338
-   Mendeley Data: https://data.mendeley.com/datasets/fywnj597d8/2

**ERRORS**

-   `acc2` in installation 3 is pure noice, as the sensor is damaged. DO NOT USE!

**Notes:**

-   Sampling frequency 3012 Hz
-   `acc4` seems to work the best


In [11]:
AGFD_raw_data_dir = Path.cwd().parent.parent.parent / "data" / "mendeley_AGFD3kHz"
AGFD_processed_data_dir = Path.cwd().parent.parent.parent / "data" / "ROTOMATE" / "AGFD"
AGFD_processed_data_dir.mkdir(parents=True, exist_ok=True)  # Ensure directory exists

### Processing


In [ ]:
AGFD_sampling_rate = 3012  # Hz

AGFD_load_map = {
    "0_12Nm": 1,
    "0_71Nm": 6,
    "1_31Nm": 11,
}

# S - Small, L - Large
AGFD_fault_map = {
    "Mild_Wear_GP7": ("S", "wear"),
    "Severe_Wear_GP2": ("L", "wear"),
    "Mild_TFF_GP9": ("S", "crack"),
    "Severe_TFF_GP5": ("L", "crack"),
    "Mild_Pitting_GP1": ("S", "pitting"),
    "Severe_Pitting_GP6": ("L", "pitting"),
}

# Raw data
files = AGFD_raw_data_dir.glob("**/*.csv")

# Go through files
AGFD_measurements = []
for f in files:
    print(f)

    # Get measurement specs
    p = f.parts

    speed_nominal = int(p[6].replace("RPM", ""))
    load_nominal = AGFD_load_map[p[7]]

    if p[8] == "Healthy":
        fault = "healthy"
        severity = "-"
        installation = 0
        GP = int(p[9][2])
    elif p[8] == "Faulty":
        installation = int(p[9][-1])
        severity, fault = AGFD_fault_map.get(
            p[10].replace(".csv", ""), (None, None)
        )  # Fault types we don't care about not in map
        GP = 0
    else:
        raise ValueError(f"Unknown type directory: {p[8]}")

    # Skip the faults we don't want
    if fault is None:
        continue

    # SIGNAL
    ##

    # Read CSV
    df = pl.scan_csv(
        f, separator=",", has_header=True, schema_overrides={"enc4_ang": pl.Float64}
    )
    # Remove index row
    df = df.select(
        pl.exclude(
            [
                "",
                # "time",
                "enc1_ang",
                "enc2_ang",
                "enc3_ang",
                # "enc4_ang",
                "enc5_ang",
            ]
        )
    ).collect()

    # These sensors not used in this study
    # df = df.select(
    #     pl.exclude(
    #         [
    #         ]
    #     ),
    # )

    # Installation 3 special case
    if installation == 3:
        # Cut first and last 20%, which has already been done for other installations
        df = df[int(len(df) * 0.2) : int(len(df) * 0.8)]

    # TODO: Save real average torque at faulty gear (Need to find out multiplier, because neither Torq1 or Torq2 are on the right shaft)
    # load = df["Torq1"].mean()

    # Some measurements have glitches in time
    # FIXME: Not working currently (also `time` not used)
    # df = df.with_columns([pl.col("time").map_elements(AGFD_fix_time)])

    # Enc4 runs the wrong way in most files
    if df["enc4_ang"][0] > df["enc4_ang"][1]:
        print("Inverting enc4_ang")
        df = df.with_columns([pl.col("enc4_ang") * (-1)])

    # Convert to tach signal
    angle = df["enc4_ang"].to_numpy() % 360
    key_timing = np.where(np.logical_and(angle[:-1] > 350, angle[1:] < 5))[0]
    tach = np.zeros_like(angle)
    np.put(tach, key_timing, 2)

    # Drop the now unnecessary enc4_ang
    df = df.with_columns([pl.Series("tach", tach)])
    # df = df.select(pl.exclude(["enc4_ang"]))

    # Calculate real speed
    speed = calc_speed(tach, sampling_freq=AGFD_sampling_rate, key_pulse_ref_height=1)

    # Sometimes last value is NaN, which causes problems later
    if any([df[s][-1] is None for s in ["acc1", "acc2", "acc3", "acc4"]]):
        df = df[:-1]

    # Flip torques and standardize naming
    df = df.rename({"Torq2": "torque1", "Torq1": "torque2"})

    # Cast float 64 columns to float 32
    # * Conversion done because final computations are done with float32 anyway
    float64_cols = [
        col for col, dtype in zip(df.columns, df.dtypes) if dtype == pl.Float64
    ]
    df = df.with_columns([pl.col(col).cast(pl.Float32) for col in float64_cols])

    df = pl.DataFrame(
        {
            "fault": fault,
            "severity": severity,
            "installation": installation,  # FIXME: Convert into i4
            "GP": GP,
            "speed_nominal": speed_nominal,
            "load_nominal": load_nominal,
            "speed": speed,  # FIXME: convert to float32/16
            # "load": load # TODO: See above
            "acc1": df["acc1"],
            "acc2": df["acc2"],
            "acc3": df["acc3"],
            "acc4": df["acc4"],
            # "torque1": df["torque1"],
            # "torque2": df["torque2"],
            "tach": df["tach"],
            "enc4_angle": df["enc4_ang"],
            "time": df["time"],
        }
    )

    AGFD_measurements.append(df)


AGFD_measurements = pl.concat(AGFD_measurements)
AGFD_measurements.head(3)

/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/0_71Nm/Healthy/GP9.csv
Inverting enc4_ang
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/0_71Nm/Healthy/GP3.csv
Inverting enc4_ang
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/0_71Nm/Healthy/GP2.csv
Inverting enc4_ang
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/0_71Nm/Healthy/GP1.csv
Inverting enc4_ang
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/0_71Nm/Healthy/GP5.csv
Inverting enc4_ang
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/0_71Nm/Healthy/GP4.csv
Inverting enc4_ang
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/0_71Nm/Healthy/GP6.csv
Inverting enc4_ang
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/0_71Nm/Healthy/GP7.csv
Inverting enc4_ang
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/0_71Nm/Faulty/Inst_1/Severe_TFF_GP5.csv
Inverting enc4_ang
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/0_71Nm/Faulty/Inst_1/Mild_Pitting_GP1.csv
Inverting enc4_ang
/Users/hamalaa14/git/dat

fault,severity,installation,GP,speed_nominal,load_nominal,speed,acc1,acc2,acc3,acc4,tach,enc4_angle,time
str,str,i32,i32,i32,i32,f64,f32,f32,f32,f32,f32,f32,f32
"""healthy""","""-""",0,9,1000,6,333.995517,14.651109,-38.778648,-6.914052,-5.760087,0.0,-0.0,0.0
"""healthy""","""-""",0,9,1000,6,333.995517,79.047661,47.732231,1.118753,-4.851206,0.0,0.666,0.000332
"""healthy""","""-""",0,9,1000,6,333.995517,26.196007,36.811794,12.214314,30.96377,0.0,1.314,0.000664


### Write & Read

#### Parquet

Simple and works when you want to read full data.


In [18]:
# Write

# Compressed by default
AGFD_measurements.write_parquet(
    AGFD_processed_data_dir / "AGFD.parquet",
)

# NOTE: To Arrow / Feather
# AGFD_measurements.write_ipc(
#     AGFD_processed_data_dir / "AGFD.arrow",
#     compression="zstd",  # Drops file size significantly (~25 Gb to ~3 Gb)
# )

In [ ]:
# Read

AGFD_df_parquet = pl.read_parquet(
    AGFD_processed_data_dir / "AGFD.parquet",
)

# NOTE: With Pandas
# AGFD_df_parquet = pd.read_parquet(
#     AGFD_processed_data_dir / "AGFD.parquet",
# )

# NOTE: From Arrow / Feather
# AGFD_df_arrow = pl.read_ipc(
#     AGFD_processed_data_dir / "AGFD.arrow",
#     memory_map=False,  # Doesn't work with compressed data
# )

# NOTE: From Arrow / Feather with Pandas
# AGFD_df_arrow = pd.read_feather(
#     AGFD_processed_data_dir / "AGFD_gear_crack.arrow",
# )

#### Parquet Hive

Fast to read when you only want part of the data.


In [19]:
# Write

AGFD_measurements.write_parquet(
    AGFD_processed_data_dir / "AGFD_hive",
    partition_by=["speed_nominal", "load_nominal"],
)

In [ ]:
# Read


def read_AGFD_hive(
    data_location,
    fault=None,
    severity=None,
    installation=None,
    GP=None,
    speed_nominal=None,
    load_nominal=None,
    drop_nominal=True,
):
    """
    data_location: Path object pointing to the hive data directory
    severity options: "-", "S", "L"
    speed_nominal_options: 250, 500, 750, 1000, 1500
    load_nominal_options: 1, 6, 11 (%)
    """

    collect = True
    if isinstance(data_location, pl.DataFrame):
        df_hive = data_location
        collect = False
    elif isinstance(data_location, Path):
        hive_files = list(data_location.glob("**/*.parquet"))
        df_hive = pl.scan_parquet(hive_files, hive_partitioning=True)
    else:
        raise ValueError("data_location must be a polars DataFrame or a Path object")

    # Get only required partitions
    if fault is not None:
        df_hive = df_hive.filter(
            pl.col("fault") == fault,
        )

    if severity is not None:
        df_hive = df_hive.filter(
            pl.col("severity") == severity,
        )

    if GP is not None:
        df_hive = df_hive.filter(
            pl.col("GP") == GP,
        )

    if installation is not None:
        df_hive = df_hive.filter(
            pl.col("installation") == installation,
        )

    if speed_nominal is not None:
        df_hive = df_hive.filter(
            pl.col("speed_nominal") == speed_nominal,
        )

    if load_nominal is not None:
        df_hive = df_hive.filter(
            pl.col("load_nominal") == load_nominal,
        )

    # Drop nominal values
    if drop_nominal:
        df_hive = df_hive.select(pl.exclude(["load_nominal", "speed_nominal"]))

    if collect:
        df_hive = df_hive.collect()

    return df_hive

In [ ]:
# Read everything

read_AGFD_hive(AGFD_processed_data_dir / "AGFD_hive").head(3)

In [ ]:
# Get one thing

read_AGFD_hive(
    AGFD_processed_data_dir / "AGFD_hive",
    fault="crack",
    severity="L",
    installation=1,
    speed_nominal=1000,
    load_nominal=1,
).head(3)

In [ ]:
# Loop through all

# Faults
for fault in ["crack", "wear", "pitting"]:
    for severity in ["S", "L"]:
        for speed_nominal in [250, 500, 750, 1000, 1250, 1500]:
            for load_nominal in [1, 6, 11]:
                for installation in [1, 2, 3]:
                    measurement = read_AGFD_hive(
                        AGFD_processed_data_dir / "AGFD_hive",
                        fault=fault,
                        severity=severity,
                        installation=installation,
                        speed_nominal=speed_nominal,
                        load_nominal=load_nominal,
                        drop_nominal=False,
                    )
                # NOTE: Do something with `measurement` here

# Healthy
for GP in [1, 2, 3, 4, 5, 6, 7, 9]:
    for speed_nominal in [250, 500, 750, 1000, 1250, 1500]:
        for load_nominal in [1, 6, 11]:
            measurement = read_AGFD_hive(
                AGFD_processed_data_dir / "AGFD_hive",
                fault="healthy",
                GP=GP,
                speed_nominal=speed_nominal,
                load_nominal=load_nominal,
                drop_nominal=False,
            )
        # NOTE: Do something with `measurement` here